# Demo 3.6 — Temperature reshapes the distribution
<a id="top"></a>

```yaml
title: Demo 3.6 — Temperature reshapes the distribution
keywords: temperature, sampling, softmax, top-p, top-k, variance, gpt2, claude
estimated duration: 14
```

> **Lesson L01**, the fifth demo in the chain. Written lecture [L0102_lecture.md](L0102_lecture.md) §5.
>
> ⚠️ **Section 2 is live**: it calls the Anthropic API and needs `ANTHROPIC_API_KEY`.
> Section 1 is the offline GPT-2 reshape and runs without a key.

**Contents**

1. [Temperature reshapes the same distribution (offline)](#1-temperature-reshapes-the-same-distribution-offline)
2. [Same prompt, different temperature (live Claude)](#2-same-prompt-different-temperature-live-claude)
3. [Low variance is not zero variance](#3-low-variance-is-not-zero-variance)

In [ ]:
from __future__ import annotations

from collections import Counter

import torch
import transformers
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

transformers.logging.set_verbosity_error()  # quiet advisory HF logs in committed output

# Offline: reuse the exact GPT-2 distribution from Demo 1 / Demo 3.5 so temperature has
# something concrete to reshape.
TOKENIZER = GPT2TokenizerFast.from_pretrained("gpt2")
MODEL = GPT2LMHeadModel.from_pretrained("gpt2").eval()
PROMPT = "Water is made of hydrogen and"

inputs = TOKENIZER(PROMPT, return_tensors="pt")
with torch.no_grad():
    LOGITS = MODEL(**inputs).logits[0, -1]  # raw scores for the next token


def reshaped_top(temperature: float, k: int = 4) -> list[tuple[str, float]]:
    """Apply temperature to the logits, softmax, and return the top-k (piece, prob)."""
    probs = torch.softmax(LOGITS / temperature, dim=-1)
    top = torch.topk(probs, k)
    return [
        (TOKENIZER.decode([int(i)]), float(p)) for p, i in zip(top.values, top.indices, strict=True)
    ]

## 1. Temperature reshapes the same distribution (offline)

Temperature scales the logits *before* the softmax. Low temperature **sharpens** (the top token
dominates); high temperature **flattens** (more candidates become competitive). Same model, same
prompt — only the knob changes.

In [ ]:
for temp in (0.5, 1.0, 2.0):
    pairs = ", ".join(f"{piece!r}={p:.3f}" for piece, p in reshaped_top(temp))
    print(f"T={temp}:  {pairs}")

At `T=0.5`, ` oxygen` sharpens toward ~0.73; at `T=2.0` it flattens toward ~0.03 and the field
levels out. Temperature does not change what the model *knows* — only how aggressively the
sampler commits to the top token.

**Now draw from it.** The table above is the *distribution*; sampling actually **picks** a token
from it. Below you take 12 draws at a fixed temperature — offline, no API key — and tally them, so
"same temperature, repeated draws" is concrete rather than described. `torch.manual_seed` keeps
the committed run reproducible.

In [ ]:
def sample_draws(temperature: float, n: int, seed: int = 0) -> list[str]:
    """Draw ``n`` next-token samples from the temperature-reshaped distribution.

    Where ``reshaped_top`` shows the probabilities, this actually *samples* — so
    repeated draws at one temperature reveal how much the sampler commits.
    """
    torch.manual_seed(seed)  # reproducible committed output
    probs = torch.softmax(LOGITS / temperature, dim=-1)
    ids = torch.multinomial(probs, num_samples=n, replacement=True)
    return [TOKENIZER.decode([int(i)]) for i in ids]


def show_draws(temperature: float, n: int = 12) -> None:
    counts = Counter(sample_draws(temperature, n))
    tally = ", ".join(f"{piece!r}={count}" for piece, count in counts.most_common())
    print(f"T={temperature}  ({n} draws): {tally}")


show_draws(0.5)
show_draws(2.0)

Same distribution, same temperature — now it's the *sampler's* behaviour on show. At `T=0.5`,
nine of the twelve draws are ` oxygen` (the rest ` helium`): low temperature means repeated draws
mostly agree. At `T=2.0`, all twelve differ — the flattened distribution lets the long tail
through, so every draw is a fresh gamble. That is exactly what the live Claude calls in section 2
demonstrate, minus the API key.

[↑ Back to top](#top)

## 2. Same prompt, different temperature (live Claude)

Now try the same idea against the anchor model. **This section needs `ANTHROPIC_API_KEY`.** The
client is constructed here (not in setup) so section 1 runs without a key.

> You'll call `client.achat(...)` — the **async** (awaitable) twin of `client.chat(...)`. Every
> `PotatoLLMClient` offers both; the course defaults to the async one, so you `await` model calls
> from the start (a notebook cell can `await` at top level). *Why* async is the default — running
> many calls at once instead of one-at-a-time — is the K05 prework's "why async for agents."

In [ ]:
from fluffy_potato_curriculum.potato_llm import AnthropicClient, Message, PotatoLLMClient

client: PotatoLLMClient = AnthropicClient()  # raises a clear error if the key is missing
COFFEE = "Suggest a name for a coffee shop on a rainy street in Seattle. Just the name, one or two words."


async def sample_n(temperature: float, n: int) -> list[str]:
    """Draw n one-shot samples at a fixed temperature, awaiting each reply."""
    replies = [
        await client.achat([Message.user(COFFEE)], max_tokens=16, temperature=temperature)
        for _ in range(n)
    ]
    return [reply.text.strip() for reply in replies]


print("temperature=0 (near-identical):")
for line in await sample_n(0.0, 5):
    print("  ", line)
print("\ntemperature=1 (varied):")
for line in await sample_n(1.0, 5):
    print("  ", line)

## 3. Low variance is not zero variance

- **Temperature 0 is *low* variance, not *zero*.** Floating-point non-determinism, server-side
  batching, and tie-breaking leak through. Run the `temperature=0` block a few more times and you
  may catch one that differs — don't promise yourself "deterministic."
- **Other knobs:** `top_p` (nucleus) samples from the smallest set of tokens summing to `p`;
  `top_k` samples from the `k` most likely. They *restrict the candidate set*; temperature
  *reshapes the distribution*. You won't tune them in this course.

| Task                                            | Temperature | Why |
| ----------------------------------------------- | ----------- | --- |
| Classification / extraction / structured output | low (≈0)    | one repeatable, most-likely answer |
| Tool routing (preview of L07)                   | low (≈0)    | a "creative" tool pick is just a bug |
| Brainstorming / creative writing                | ≈0.7–1.0    | variety is the point |

[↑ Back to top](#top)